### K-Meansとは
- K-Meansは、データをK個のクラスターに分ける手法。
- K-Meansは教師なし学習。正解ラベルを使わず、データの構造だけからグループを発見する。
- K個の重心（セントロイド）をランダムに初期化し、各データ点を最も近い重心のクラスターに割り当て、各クラスターの重心を更新し、これを収束するまで繰り返す。

||内容|
|----|----|
|手法|教師なし学習（クラスタリング）|
|目的関数|クラスター内慣性（WCSS）の最小化|
|重心の更新|クラスター内データ点の平均|
|収束|割り当てが変化しなくなるまで繰り返す|
|注意点|局所解に陥る可能性あり、初期値依存|


In [4]:
# ライブラリでの実装
# データセットのラベルを使わず、特徴量だけでクラスタリングを行う

import numpy as np
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target

model = KMeans(n_clusters=3, random_state=42, n_init=10)
model.fit(X)
labels = model.labels_

print(f"クラスター割り当て（先頭10件）: {labels[:10]}")
print(f"正解ラベル（先頭10件）        : {y[:10]}")

クラスター割り当て（先頭10件）: [1 1 1 1 1 1 1 1 1 1]
正解ラベル（先頭10件）        : [0 0 0 0 0 0 0 0 0 0]


In [3]:
# scratchでのK-Meansの実装

import numpy as np
from sklearn.datasets import load_iris


class KMeans:
    def __init__(self, n_clusters=3, n_epochs=100, random_state=None):
        self.n_clusters = n_clusters
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None

    def fit(self, X):
        rng = np.random.default_rng(self.random_state)
        indices = rng.choice(len(X), self.n_clusters, replace=False)
        self.centroids = X[indices].copy()

        for _ in range(self.n_epochs):
            labels = self._assign(X)
            new_centroids = np.array([
                X[labels == k].mean(axis=0) for k in range(self.n_clusters)
            ])
            if np.allclose(self.centroids, new_centroids):
                break
            self.centroids = new_centroids

        self.labels_ = self._assign(X)

    def _assign(self, X):
        distances = np.array([
            np.sum((X - centroid) ** 2, axis=1) for centroid in self.centroids
        ])
        return np.argmin(distances, axis=0)

    def predict(self, X):
        return self._assign(X)


# 実行
iris = load_iris()
X, y = iris.data, iris.target

model = KMeans(n_clusters=3, n_epochs=100, random_state=42)
model.fit(X)

print(f"クラスター割り当て（先頭10件）: {model.labels_[:10]}")
print(f"正解ラベル（先頭10件）        : {y[:10]}")

クラスター割り当て（先頭10件）: [1 1 1 1 1 1 1 1 1 1]
正解ラベル（先頭10件）        : [0 0 0 0 0 0 0 0 0 0]


初期化

`rng.choice` でランダムにK個のデータ点を選び、初期重心とします。`replace=False` で同じ点が選ばれないようにしています。

`_assign` メソッド

各重心とX全点の距離をまとめて計算し、`np.argmin(axis=0)` で各データ点が最も近い重心のインデックスを返します。

収束判定

`np.allclose` で重心の変化がほぼゼロになったかを確認します。浮動小数点誤差を考慮して、厳密に等しいかどうかの判定は避けています。